In [1]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv("wfp_food_prices_uga.csv", parse_dates=["date"])

# Filter to Maize in Owino (Kampala)
owino = df[(df["commodity"] == "Maize") & (df["market"] == "Owino")].copy()
owino = owino.sort_values("date").reset_index(drop=True)

print(f"Rows: {len(owino)}")
print(f"Date range: {owino['date'].min().date()} to {owino['date'].max().date()}")
owino[["date", "price"]].head(8)


Rows: 202
Date range: 2006-02-15 to 2022-05-15


,date,price
0,2006-02-15,347.23
1,2006-03-15,394.36
2,2006-06-15,395.99
3,2006-07-15,300.63
4,2006-08-15,216.57
5,2006-09-15,267.71
6,2006-11-15,303.11
7,2007-03-15,253.31


## Feature Engineering Plan

Each row will represent one month. The model sees:

| Feature | Description | Why useful |
|---------|-------------|------------|
| `price_lag_1` | Price last month | Strong predictor — prices are sticky |
| `price_lag_2` | Price 2 months ago | Captures short-term momentum |
| `price_lag_3` | Price 3 months ago | Captures short-term momentum |
| `rolling_mean_3` | Avg price over last 3 months | Smooths noise, shows recent trend |
| `rolling_mean_6` | Avg price over last 6 months | Captures medium-term trend |
| `month` | Calendar month (1–12) | Captures harvest seasonality |
| `year` | Calendar year | Captures long-term inflation |

**Target:** `price` shifted forward by 1 month — i.e., "next month's price"

**Important — avoid data leakage:**
All rolling/lag calculations use `.shift(1)` before `.rolling()` so we never include
the current month's price in the rolling window. The model only sees the past.


In [2]:
def build_features(price_series: pd.Series, dates: pd.Series) -> pd.DataFrame:
    df = pd.DataFrame({"date": dates, "price": price_series}).reset_index(drop=True)

    # Lag features: look back N months
    df["price_lag_1"] = df["price"].shift(1)
    df["price_lag_2"] = df["price"].shift(2)
    df["price_lag_3"] = df["price"].shift(3)

    # Rolling averages over lagged series (shift first to avoid leakage)
    df["rolling_mean_3"] = df["price"].shift(1).rolling(window=3).mean()
    df["rolling_mean_6"] = df["price"].shift(1).rolling(window=6).mean()

    # Calendar features
    df["month"] = df["date"].dt.month
    df["year"] = df["date"].dt.year

    # Target: next month's price
    df["target"] = df["price"].shift(-1)

    # Drop rows where we can't compute all features or target
    df = df.dropna().reset_index(drop=True)

    return df

features_df = build_features(owino["price"], owino["date"])
print(f"Shape after feature engineering: {features_df.shape}")
print(f"Date range: {features_df['date'].min().date()} to {features_df['date'].max().date()}")
features_df.head(4)


Shape after feature engineering: (195, 10)
Date range: 2006-11-15 to 2022-04-15


,date,price,price_lag_1,price_lag_2,price_lag_3,rolling_mean_3,rolling_mean_6,month,year,target
0,2006-11-15,303.11,267.71,216.57,300.63,261.636667,320.415000,11,2006,253.31
1,2007-03-15,253.31,303.11,267.71,216.57,262.463333,313.061667,3,2007,241.13
2,2007-04-15,241.13,253.31,303.11,267.71,274.710000,289.553333,4,2007,249.62
3,2007-05-15,249.62,241.13,253.31,303.11,265.850000,263.743333,5,2007,214.87


In [4]:
row = features_df.iloc[0]
print(f"Row 0 date: {row['date'].date()}")
print(f"Row 0 price_lag_1: {row['price_lag_1']:.2f}")

prev_idx = owino[owino["date"] < row["date"]].index[-1]
print(f"Actual previous month price: {owino.loc[prev_idx, 'price']:.2f}")
print("Match:", abs(row['price_lag_1'] - owino.loc[prev_idx, 'price']) < 0.01)

Row 0 date: 2006-11-15
Row 0 price_lag_1: 267.71
Actual previous month price: 267.71
Match: True


In [5]:
FEATURE_COLS = [
    "price_lag_1", "price_lag_2", "price_lag_3",
    "rolling_mean_3", "rolling_mean_6",
    "month", "year"
]

train = features_df[features_df["year"] <= 2019].copy()
test  = features_df[features_df["year"] >= 2020].copy()

print(f"Train: {len(train)} rows  ({train['year'].min()}–{train['year'].max()})")
print(f"Test:  {len(test)} rows  ({test['year'].min()}–{test['year'].max()})")
print()
print("Train target stats:")
print(train["target"].describe().round(0))


Train: 175 rows  (2006–2019)
Test:  20 rows  (2020–2022)

Train target stats:
count     175.0
mean      796.0
std       298.0
min       204.0
25%       600.0
50%       774.0
75%       999.0
max      1655.0
Name: target, dtype: float64


## Why we split by time (not randomly)

In a normal ML problem you might randomly shuffle rows into train/test.
For time series, that would be **data leakage** — the model would be trained on
data from 2021 and tested on data from 2010. That's cheating: in the real world
you can only predict the future, not the past.

**Rule:** the test set must always be the most recent period.
We use 2020–2022 as the test set because it contains COVID-era price shocks,
which are a realistic challenge for the model.


In [7]:
os.makedirs("data", exist_ok=True)
features_df.to_csv("data/maize_owino_features.csv", index=False)
print("Saved to data/maize_owino_features.csv")


Saved to data/maize_owino_features.csv
